# Colab Drive
Hệ thống tự động hóa tải và đồng bộ tệp tin đa nguồn vào Google Drive trên nền tảng Google Colab.

Chức năng chính:
- Hỗ trợ đa giao thức: HTTP/HTTPS direct link, HLS (.m3u8), DASH (.mpd), BitTorrent/Magnet, và hơn 1.800 nền tảng trực tuyến qua yt-dlp.
- Hỗ trợ mạng máy chủ ủy danh (Proxy HTTP/HTTPS/SOCKS5) xoay tua linh hoạt, phân tách bằng dấu phẩy, khoảng trắng, chấm phẩy hoặc xuống dòng.
- Nạp Cookie (chuỗi Netscape hoặc tệp cookies.txt) để mở khóa nội dung 4K, 1080p Premium và tài khoản VIP/giới hạn.
- Tự động duy trì phiên kết nối Colab (Anti-Disconnect).
- Tự động kiểm tra dung lượng Google Drive trước và sau khi lưu trữ.
- Tự động giải nén các định dạng lưu trữ (ZIP, RAR, 7Z, TAR) tốc độ cao qua p7zip.
- Tự động gửi thông báo báo cáo tiến trình qua Telegram Bot.
- Tự động tối ưu băng thông tải xuống bằng aria2c đa luồng và chuyển đổi phương thức thông minh khi gặp sự cố.

Hướng dẫn vận hành: Thực thi tuần tự các bước từ Bước 1 đến Bước 3.


In [ ]:
#@title Bước 1: Thiết lập môi trường và cài đặt phụ thuộc
#@markdown Khởi tạo môi trường, cài đặt yt-dlp, ffmpeg, aria2c, p7zip và kích hoạt chống ngắt kết nối.

import os
import sys
from datetime import datetime, timezone, timedelta
from IPython.display import display, Javascript

VN_TZ = timezone(timedelta(hours=7))

def log(tag, msg, level="INFO"):
    colors = {
        "SYSTEM": "\033[36m",
        "SUCCESS": "\033[32m",
        "INFO": "\033[34m",
        "WARN": "\033[33m",
        "ERROR": "\033[31m",
        "DRIVE": "\033[35m",
        "FETCH": "\033[94m",
        "TRANSFER": "\033[33m",
        "EXTRACT": "\033[96m",
        "NOTIFY": "\033[92m"
    }
    reset = "\033[0m"
    dim = "\033[90m"
    ts = datetime.now(VN_TZ).strftime("%H:%M:%S")
    tag_color = colors.get(level, "\033[37m")
    print(f"{dim}{ts}{reset} {tag_color}[{tag}]{reset} {msg}")

log("SYSTEM", "Bắt đầu khởi tạo môi trường hệ thống...", "SYSTEM")

# Kích hoạt script tự động duy trì kết nối trình duyệt (Anti-Disconnect)
display(Javascript('''
function KeepClicking(){
   console.log("[Colab Drive] Anti-disconnect heartbeat pulse sent");
   document.querySelector("colab-connect-button")?.shadowRoot?.querySelector("#connect")?.click();
}
setInterval(KeepClicking, 60000);
'''))
log("SYSTEM", "Đã kích hoạt cơ chế tự động duy trì kết nối phiên làm việc (Anti-Disconnect).", "SYSTEM")

log("SYSTEM", "Cài đặt và cập nhật gói yt-dlp mới nhất...", "INFO")
os.system("pip install --upgrade --no-cache-dir -q yt-dlp")

log("SYSTEM", "Cài đặt aria2c, ffmpeg, p7zip-full và unrar qua trình quản lý APT...", "INFO")
os.system("apt-get update -y -q && apt-get install -y -q ffmpeg aria2 p7zip-full unrar")

log("SUCCESS", "Thiết lập môi trường và phụ thuộc hoàn tất.", "SUCCESS")



In [ ]:
#@title Bước 2: Xác thực và kết nối Google Drive
#@markdown Liên kết phân vùng lưu trữ Google Drive và kiểm tra dung lượng khả dụng.

import os
import shutil
from datetime import datetime, timezone, timedelta

VN_TZ = timezone(timedelta(hours=7))

def log(tag, msg, level="INFO"):
    colors = {
        "SYSTEM": "\033[36m",
        "SUCCESS": "\033[32m",
        "INFO": "\033[34m",
        "WARN": "\033[33m",
        "ERROR": "\033[31m",
        "DRIVE": "\033[35m",
        "FETCH": "\033[94m",
        "TRANSFER": "\033[33m",
        "EXTRACT": "\033[96m",
        "NOTIFY": "\033[92m"
    }
    reset = "\033[0m"
    dim = "\033[90m"
    ts = datetime.now(VN_TZ).strftime("%H:%M:%S")
    tag_color = colors.get(level, "\033[37m")
    print(f"{dim}{ts}{reset} {tag_color}[{tag}]{reset} {msg}")

log("DRIVE", "Yêu cầu quyền truy cập phân vùng Google Drive...", "DRIVE")
try:
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists('/content/drive/MyDrive'):
        log("SUCCESS", "Kết nối Google Drive thành công.", "SUCCESS")
        log("DRIVE", "Điểm gắn lưu trữ: /content/drive/MyDrive/", "INFO")
        
        # Kiểm tra dung lượng lưu trữ của Google Drive
        total, used, free = shutil.disk_usage('/content/drive/MyDrive')
        total_gb = total / (1024**3)
        used_gb = used / (1024**3)
        free_gb = free / (1024**3)
        log("DRIVE", f"Dung lượng Google Drive: Đã dùng {used_gb:.2f} GB / Còn trống {free_gb:.2f} GB (Tổng: {total_gb:.2f} GB).", "DRIVE")
        if free_gb < 1.0:
            log("WARN", f"Cảnh báo: Dung lượng trống trên Google Drive chỉ còn {free_gb:.2f} GB.", "WARN")
    else:
        log("WARN", "Đã gắn kết phân vùng nhưng chưa nhận diện được thư mục MyDrive.", "WARN")
except Exception as e:
    log("ERROR", f"Kết nối Google Drive thất bại: {e}", "ERROR")



In [ ]:
#@title Bước 3: Tải xuống và đồng bộ vào Google Drive
#@markdown Nhập liên kết cần tải về và cấu hình các tùy chọn bổ trợ nếu cần.

LINK_CAN_TAI = "" #@param {type:"string"}
THU_MUC_DRIVE = "/content/drive/MyDrive/colab-drive" #@param {type:"string"}
TEN_FILE_TU_DAT = "" #@param {type:"string"}
DANH_SACH_PROXY = "" #@param {type:"string"}
REFERER_VUOT_CHAN = "" #@param {type:"string"}
NOI_DUNG_COOKIE = "" #@param {type:"string"}
TU_DONG_GIAI_NEN = False #@param {type:"boolean"}
TELEGRAM_BOT_TOKEN = "" #@param {type:"string"}
TELEGRAM_CHAT_ID = "" #@param {type:"string"}

import os
import re
import glob
import json
import shutil
import random
import urllib.request
import urllib.parse
from datetime import datetime, timezone, timedelta
import yt_dlp

VN_TZ = timezone(timedelta(hours=7))

def log(tag, msg, level="INFO"):
    colors = {
        "SYSTEM": "\033[36m",
        "SUCCESS": "\033[32m",
        "INFO": "\033[34m",
        "WARN": "\033[33m",
        "ERROR": "\033[31m",
        "DRIVE": "\033[35m",
        "FETCH": "\033[94m",
        "TRANSFER": "\033[33m",
        "EXTRACT": "\033[96m",
        "NOTIFY": "\033[92m"
    }
    reset = "\033[0m"
    dim = "\033[90m"
    ts = datetime.now(VN_TZ).strftime("%H:%M:%S")
    tag_color = colors.get(level, "\033[37m")
    print(f"{dim}{ts}{reset} {tag_color}[{tag}]{reset} {msg}")

def send_telegram(token, chat_id, text):
    if not token.strip() or not chat_id.strip():
        return
    try:
        url = f"https://api.telegram.org/bot{token.strip()}/sendMessage"
        payload = json.dumps({"chat_id": chat_id.strip(), "text": text}).encode('utf-8')
        req = urllib.request.Request(url, data=payload, headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req, timeout=10) as resp:
            if resp.status == 200:
                log("NOTIFY", "Đã gửi thông báo báo cáo tiến trình qua Telegram.", "NOTIFY")
    except Exception as e:
        log("WARN", f"Không thể gửi thông báo Telegram: {e}", "WARN")

def parse_proxies(proxy_raw):
    proxies = []
    if not proxy_raw or not proxy_raw.strip():
        return proxies
    raw_clean = proxy_raw.replace(',', '\n').replace(';', '\n').replace('|', '\n').replace(' ', '\n')
    for line in raw_clean.split('\n'):
        line = line.strip()
        if not line:
            continue
        parts = line.split(':')
        if len(parts) == 4:
            ip, port, user, pwd = parts
            proxies.append(f"http://{user}:{pwd}@{ip}:{port}")
        elif len(parts) == 2:
            ip, port = parts
            proxies.append(f"http://{ip}:{port}")
        elif line.startswith(('http://', 'https://', 'socks5://', 'socks4://')):
            proxies.append(line)
    return proxies

def mask_proxy(p_str):
    return re.sub(r'//.*?:.*?@', '//***:***@', p_str)

def get_unique_filename(dest_dir, filename):
    base, ext = os.path.splitext(filename)
    counter = 1
    new_filename = filename
    while os.path.exists(os.path.join(dest_dir, new_filename)):
        new_filename = f"{base}_{counter}{ext}"
        counter += 1
    return new_filename

if not LINK_CAN_TAI.strip():
    log("ERROR", "Chưa có thông tin liên kết đầu vào.", "ERROR")
    raise ValueError("Vui lòng nhập liên kết hợp lệ tại ô LINK_CAN_TAI.")

raw_urls = LINK_CAN_TAI.replace(',', '\n').replace(';', '\n')
urls = [u.strip() for u in raw_urls.split('\n') if u.strip()]

proxies = parse_proxies(DANH_SACH_PROXY)
temp_workspace = "/content/temp_download"
cookie_file_path = None

if os.path.exists(temp_workspace):
    shutil.rmtree(temp_workspace, ignore_errors=True)
os.makedirs(temp_workspace, exist_ok=True)
os.makedirs(THU_MUC_DRIVE, exist_ok=True)

# Kiểm tra dung lượng Google Drive trước khi tiến hành
if os.path.exists('/content/drive/MyDrive'):
    _, _, drive_free = shutil.disk_usage('/content/drive/MyDrive')
    free_gb = drive_free / (1024**3)
    log("DRIVE", f"Kiểm tra bộ nhớ: Google Drive hiện còn trống {free_gb:.2f} GB.", "DRIVE")
    if free_gb < 0.5:
        log("WARN", "Cảnh báo: Bộ nhớ Google Drive còn dưới 500 MB, tiến trình có thể bị gián đoạn.", "WARN")

# Xử lý Cookie nạp vào
if NOI_DUNG_COOKIE.strip():
    cookie_input = NOI_DUNG_COOKIE.strip()
    if os.path.isfile(cookie_input):
        cookie_file_path = cookie_input
        log("SYSTEM", f"Sử dụng tệp Cookie xác thực: {cookie_file_path}", "INFO")
    else:
        cookie_file_path = os.path.join(temp_workspace, "auth_cookies.txt")
        with open(cookie_file_path, "w", encoding="utf-8") as cf:
            cf.write(cookie_input)
        log("SYSTEM", "Đã khởi tạo tệp Cookie xác thực từ dữ liệu đầu vào.", "INFO")

log("SYSTEM", f"Khởi chạy tiến trình xử lý cho {len(urls)} liên kết.", "SYSTEM")
if proxies:
    log("SYSTEM", f"Đã nạp thành công {len(proxies)} địa chỉ proxy (sẵn sàng xoay tua phân bổ).", "INFO")

success_count = 0
failed_count = 0
saved_files_report = []

for index, target_url in enumerate(urls, start=1):
    log("FETCH", f"[{index}/{len(urls)}] Đang xử lý: {target_url}", "FETCH")
    
    current_proxy = proxies[(index - 1) % len(proxies)] if proxies else None
    if current_proxy:
        log("SYSTEM", f"Định tuyến qua proxy ({((index - 1) % len(proxies)) + 1}/{len(proxies)}): {mask_proxy(current_proxy)}", "INFO")
        
    sub_task_dir = os.path.join(temp_workspace, f"task_{index:03d}")
    os.makedirs(sub_task_dir, exist_ok=True)
    
    is_torrent = target_url.startswith('magnet:?') or target_url.lower().endswith('.torrent') or '.torrent?' in target_url.lower()
    
    download_ok = False
    
    if is_torrent:
        log("FETCH", "Phát hiện định dạng BitTorrent/Magnet. Khởi chạy aria2c đa luồng...", "FETCH")
        trackers = [
            "udp://tracker.opentrackr.org:1337/announce",
            "udp://open.tracker.cl:1337/announce",
            "udp://9.rarbg.com:2810/announce",
            "udp://tracker.openbittorrent.com:6969/announce",
            "http://tracker.openbittorrent.com:80/announce"
        ]
        tracker_arg = f"--bt-tracker={','.join(trackers)}"
        aria2_cmd = (
            f'aria2c --seed-time=0 --summary-interval=5 --file-allocation=none '
            f'--enable-dht=true --dht-listen-port=6881 --bt-enable-lpd=true '
            f'{tracker_arg} --dir="{sub_task_dir}" "{target_url}"'
        )
        if current_proxy:
            aria2_cmd += f' --all-proxy="{current_proxy}"'
            
        ret = os.system(aria2_cmd)
        if ret == 0:
            download_ok = True
        else:
            log("ERROR", f"Tiến trình aria2c báo lỗi với mã thoát: {ret}", "ERROR")
    else:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
            'Accept': '*/*',
            'Accept-Language': 'en-US,en;q=0.9,vi;q=0.8'
        }
        if REFERER_VUOT_CHAN.strip():
            headers['Referer'] = REFERER_VUOT_CHAN.strip()
            headers['Origin'] = urllib.parse.urlsplit(REFERER_VUOT_CHAN.strip()).scheme + "://" + urllib.parse.urlsplit(REFERER_VUOT_CHAN.strip()).netloc
        elif 'seenow' in target_url.lower() or 'vtv' in target_url.lower():
            headers['Referer'] = 'https://vtvprime.vn/'
            headers['Origin'] = 'https://vtvprime.vn'

        if TEN_FILE_TU_DAT.strip():
            custom_raw = TEN_FILE_TU_DAT.strip()
            base_name, specified_ext = os.path.splitext(custom_raw)
            num_suffix = f"_{index}" if len(urls) > 1 else ""
            if specified_ext:
                out_template = os.path.join(sub_task_dir, f"{base_name}{num_suffix}{specified_ext}")
            else:
                out_template = os.path.join(sub_task_dir, f"{base_name}{num_suffix}.%(ext)s")
        else:
            out_template = os.path.join(sub_task_dir, '%(title)s.%(ext)s')

        aria2_args = ['-x16', '-s16', '-k1M', '--file-allocation=none']
        if current_proxy:
            aria2_args.append(f'--all-proxy={current_proxy}')
        if 'Referer' in headers:
            aria2_args.append(f'--header=Referer: {headers["Referer"]}')
        if 'Origin' in headers:
            aria2_args.append(f'--header=Origin: {headers["Origin"]}')

        ydl_opts = {
            'outtmpl': out_template,
            'format': 'bestvideo*+bestaudio/best',
            'merge_output_format': 'mp4',
            'quiet': False,
            'no_warnings': True,
            'retries': 10,
            'fragment_retries': 10,
            'concurrent_fragment_downloads': 8,
            'http_headers': headers,
            'external_downloader': 'aria2c',
            'external_downloader_args': aria2_args
        }
        if cookie_file_path:
            ydl_opts['cookiefile'] = cookie_file_path
        if current_proxy:
            ydl_opts['proxy'] = current_proxy

        log("FETCH", "Khởi chạy tải đa luồng tốc độ cao (yt-dlp + aria2c)...", "INFO")
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([target_url])
            download_ok = True
        except Exception as err:
            log("WARN", f"Chế độ aria2c gặp lỗi ({err}). Tự động chuyển sang bộ tải native yt-dlp...", "WARN")
            ydl_opts.pop('external_downloader', None)
            ydl_opts.pop('external_downloader_args', None)
            try:
                with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                    ydl.download([target_url])
                download_ok = True
            except Exception as err2:
                log("WARN", f"Định dạng ghép kênh không hỗ trợ ({err2}). Tự động thử định dạng tương thích 'best'...", "WARN")
                ydl_opts['format'] = 'best'
                try:
                    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                        ydl.download([target_url])
                    download_ok = True
                except Exception as err3:
                    log("WARN", f"Tải stream thất bại ({err3}). Thử nghiệm tải trực tiếp bằng aria2c...", "WARN")
                    direct_cmd = f'aria2c -x16 -s16 -k1M --file-allocation=none -d "{sub_task_dir}" "{target_url}"'
                    if current_proxy:
                        direct_cmd += f' --all-proxy="{current_proxy}"'
                    if os.system(direct_cmd) == 0:
                        download_ok = True
                    else:
                        log("ERROR", f"Không thể tải xuống tệp từ nguồn: {target_url}", "ERROR")

    # Xử lý giải nén tệp tự động nếu được bật
    archive_extensions = ('.zip', '.rar', '.7z', '.tar', '.gz', '.tar.gz', '.tgz', '.bz2')
    if download_ok and TU_DONG_GIAI_NEN:
        for root, dirs, files in os.walk(sub_task_dir):
            for f in files:
                f_path = os.path.join(root, f)
                lower_f = f.lower()
                if lower_f.endswith(archive_extensions):
                    folder_name = os.path.splitext(f)[0]
                    if folder_name.lower().endswith('.tar'):
                        folder_name = os.path.splitext(folder_name)[0]
                    extract_dest = os.path.join(sub_task_dir, f"extracted_{folder_name}")
                    os.makedirs(extract_dest, exist_ok=True)
                    log("EXTRACT", f"Đang tiến hành giải nén tệp lưu trữ: {f}...", "EXTRACT")
                    cmd_extract = f'7z x -y "{f_path}" -o"{extract_dest}" > /dev/null 2>&1'
                    status_ext = os.system(cmd_extract)
                    if status_ext == 0:
                        log("SUCCESS", f"Giải nén thành công tệp: {f}", "SUCCESS")
                        try:
                            os.remove(f_path)
                        except:
                            pass
                    else:
                        log("WARN", f"Không thể giải nén tự động tệp: {f}", "WARN")

    # Quét tệp tin đầu ra và chuyển giao lên Google Drive
    output_items = []
    for root, dirs, files in os.walk(sub_task_dir):
        for f in files:
            if not f.endswith(('.part', '.ytdl', '.aria2')):
                output_items.append(os.path.join(root, f))

    if download_ok and output_items:
        log("TRANSFER", f"Chuyển giao {len(output_items)} tệp tin vào Google Drive: {THU_MUC_DRIVE}", "TRANSFER")
        
        custom_raw = TEN_FILE_TU_DAT.strip()
        custom_base, specified_ext = os.path.splitext(custom_raw) if custom_raw else ("", "")
        num_suffix = f"_{index}" if len(urls) > 1 else ""

        for item_idx, fpath in enumerate(output_items):
            orig_filename = os.path.basename(fpath)
            orig_base, orig_ext = os.path.splitext(orig_filename)

            if custom_raw and (is_torrent or not os.path.exists(fpath)):
                final_ext = specified_ext if specified_ext else orig_ext
                part_suffix = f"_{item_idx+1}" if len(output_items) > 1 else ""
                dest_filename = f"{custom_base}{num_suffix}{part_suffix}{final_ext}"
            else:
                dest_filename = orig_filename

            target_fname = get_unique_filename(THU_MUC_DRIVE, dest_filename)
            dest_path = os.path.join(THU_MUC_DRIVE, target_fname)
            shutil.move(fpath, dest_path)
            log("SUCCESS", f"Lưu trữ thành công: {dest_path}", "SUCCESS")
            saved_files_report.append(target_fname)
        success_count += 1
    else:
        log("ERROR", f"Thất bại khi xử lý tiến trình [{index}/{len(urls)}]", "ERROR")
        failed_count += 1

try:
    shutil.rmtree(temp_workspace, ignore_errors=True)
except:
    pass

print("\n" + "="*60)
log("SYSTEM", f"Tổng kết: {success_count} thành công, {failed_count} thất bại.", "SYSTEM")
log("DRIVE", f"Thư mục lưu trữ trên Google Drive: {THU_MUC_DRIVE}", "DRIVE")
print("="*60)

# Gửi thông báo hoàn tất qua Telegram Bot nếu có cấu hình
if TELEGRAM_BOT_TOKEN.strip() and TELEGRAM_CHAT_ID.strip():
    ts_now = datetime.now(VN_TZ).strftime("%H:%M:%S - %d/%m/%Y")
    report_lines = [
        "[Colab Drive] Bao cao ket qua tien trinh:",
        f"- Thoi gian hoan tat: {ts_now}",
        f"- Tong so URL: {len(urls)}",
        f"- Thanh cong: {success_count}",
        f"- That bai: {failed_count}",
        f"- Thu muc luu tru: {THU_MUC_DRIVE}"
    ]
    if saved_files_report:
        report_lines.append(f"- Tep da luu ({len(saved_files_report)}):")
        for fn in saved_files_report[:10]:
            report_lines.append(f"  + {fn}")
        if len(saved_files_report) > 10:
            report_lines.append(f"  ... va {len(saved_files_report) - 10} tep khac")
    
    send_telegram(TELEGRAM_BOT_TOKEN, TELEGRAM_CHAT_ID, "\n".join(report_lines))

